In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\


Failed to read module file 'C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen imp

In [2]:
import pandas as pd

import torch
from models.resnet import ResnetMultilabel
from models.mobilenet import MobileNetMultilabel
from models.quant_mobilenet import load_mobilenet_v3_quant

from training.cross_validation import run_cross_val, train_model


## Running the Optimization Experiments

This section covers the model optimization experiments:
- Switching from **ResNet18** to **MobileNet V3 Small**,
- Further **Truncating** the MobileNet architecture,
- Applying **8-bit Quantization-Aware Training (QAT)** on the MobileNet model.

Each experiment is executed with **cross-validation** as described in the paper.  
Results are written to a dedicated `results/` directory and subsequently examined in the `results_analysis` folder.



In [3]:
labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")
labels_df["ClipFilenamePt"] = labels_df["clip_filename"].str.replace(".wav", ".pt", regex=False)


label_columns = ["ECHO", "HFPC", "BBPC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Verified_Dataset/spectrograms/"

results_dir = "./results/new_dataset"

In [5]:
labels_df.groupby("Site")["Boat"].value_counts()

Site  Boat
BSM   1.0      992
      0.0      943
CAC   0.0     1896
      1.0     1102
KAM   1.0     2234
      0.0     1171
RDL   0.0      348
      1.0       27
Name: count, dtype: int64

In [10]:
training_config_default = {
    "batch_size": 32,
    "lr_decay_factor": 0.5,
    "patience_lr": 2,
    # "n_epochs": 1, #100
    # "min_epochs": 0, #15
    "n_epochs": 100, #100
    "min_epochs": 10, #15
    "patience_early_stopping": 5,
    "metric_mode": "max",
    "val_metric": "f1",
}

In [5]:
import torch
torch.cuda.is_available()

True

### Resnet18


In [ ]:
run_cross_val(
    labels_df, 
    label_columns, 
    ResnetMultilabel,  
    processed_spects_dir,
    run_name="resnet",
    results_dir=results_dir,
    model_kwargs={
        "pretrained":True,
    }, 
    training_config=training_config_default,
    save_models=True,
    use_quantization=False,
    test_cols_metrics=["Site", "Boat"],
    fold_exclusive_col="labeled_snippet_filename"
)


Val Epoch: 30


100%|██████████| 78/78 [00:06<00:00, 11.56it/s]


Val Epoch: 30 Results - 
loss: 26.892, 
accuracy: {'Labels_Average': 0.9722110629081726, 'ECHO': 0.9577124714851379, 'HFPC': 0.9855014085769653, 'BBPC': 0.9826822280883789, 'Whistle': 0.9629480242729187}, 
f1: {'Labels_Average': 0.8842042684555054, 'ECHO': 0.9395509362220764, 'HFPC': 0.8500000238418579, 'BBPC': 0.8501741886138916, 'Whistle': 0.8970917463302612}, 
precision: {'Labels_Average': 0.9025516510009766, 'ECHO': 0.9477351903915405, 'HFPC': 0.8793103694915771, 'BBPC': 0.8840579986572266, 'Whistle': 0.8991031646728516}, 
recall: {'Labels_Average': 0.866992175579071, 'ECHO': 0.931506872177124, 'HFPC': 0.8225806355476379, 'BBPC': 0.818791925907135, 'Whistle': 0.8950892686843872}, 
AUC: {'Labels_Average': 0.9879193305969238, 'ECHO': 0.9907183647155762, 'HFPC': 0.9836538434028625, 'BBPC': 0.9873909950256348, 'Whistle': 0.9899142384529114}, 
exact_match: {'Labels_Average': 0.90495365858078},

Training Epoch: 31


100%|██████████| 311/311 [00:38<00:00,  8.01it/s]


Train Epoch: 31 Results - 
loss: 0.095, 
accuracy: {'Labels_Average': 0.9997734427452087, 'ECHO': 1.0, 'HFPC': 0.9996979236602783, 'BBPC': 0.9995972514152527, 'Whistle': 0.9997986555099487}, 
f1: {'Labels_Average': 0.9983233213424683, 'ECHO': 1.0, 'HFPC': 0.9969848990440369, 'BBPC': 0.9968404173851013, 'Whistle': 0.9994677901268005}, 
precision: {'Labels_Average': 0.9985740184783936, 'ECHO': 1.0, 'HFPC': 0.9979879260063171, 'BBPC': 0.9968404173851013, 'Whistle': 0.9994677901268005}, 
recall: {'Labels_Average': 0.9980730414390564, 'ECHO': 1.0, 'HFPC': 0.9959839582443237, 'BBPC': 0.9968404173851013, 'Whistle': 0.9994677901268005}, 
AUC: {'Labels_Average': 0.9999978542327881, 'ECHO': 1.0, 'HFPC': 0.9999942779541016, 'BBPC': 0.9999972581863403, 'Whistle': 0.9999997615814209}, 
exact_match: {'Labels_Average': 0.9990938305854797},

Val Epoch: 31


...

100%|██████████| 50/50 [00:05<00:00,  9.62it/s]


Test on site bsm Epoch: 0 Results - 
loss: 15.283, 
accuracy: {'Labels_Average': 0.9846129417419434, 'ECHO': 0.9695431590080261, 'HFPC': 0.9904822111129761, 'BBPC': 0.9917512536048889, 'Whistle': 0.9866751432418823}, 
f1: {'Labels_Average': 0.847484290599823, 'ECHO': 0.8883720636367798, 'HFPC': 0.8695651888847351, 'BBPC': 0.800000011920929, 'Whistle': 0.8320000171661377}, 
precision: {'Labels_Average': 0.8553147315979004, 'ECHO': 0.8925233483314514, 'HFPC': 0.8620689511299133, 'BBPC': 0.8666666746139526, 'Whistle': 0.800000011920929}, 
recall: {'Labels_Average': 0.842743992805481, 'ECHO': 0.8842592835426331, 'HFPC': 0.8771929740905762, 'BBPC': 0.7428571581840515, 'Whistle': 0.8666666746139526}, 
AUC: {'Labels_Average': 0.9943938255310059, 'ECHO': 0.9925721287727356, 'HFPC': 0.996107816696167, 'BBPC': 0.9925837516784668, 'Whistle': 0.9963115453720093}, 
exact_match: {'Labels_Average': 0.9473350048065186},
                   Filename Site  ECHO_true  ECHO_pred    ECHO_probs  \
0  BSM_201

100%|██████████| 21/21 [00:02<00:00,  9.68it/s]


Test on site cac Epoch: 0 Results - 
loss: 60.938, 
accuracy: {'Labels_Average': 0.9429223537445068, 'ECHO': 0.9391171932220459, 'HFPC': 0.9832572340965271, 'BBPC': 0.943683385848999, 'Whistle': 0.9056316614151001}, 
f1: {'Labels_Average': 0.8607894778251648, 'ECHO': 0.9224806427955627, 'HFPC': 0.8674699068069458, 'BBPC': 0.7482993006706238, 'Whistle': 0.9049080014228821}, 
precision: {'Labels_Average': 0.88736492395401, 'ECHO': 0.9520000219345093, 'HFPC': 0.9230769276618958, 'BBPC': 0.7638888955116272, 'Whistle': 0.9104938507080078}, 
recall: {'Labels_Average': 0.8364105224609375, 'ECHO': 0.8947368264198303, 'HFPC': 0.8181818127632141, 'BBPC': 0.7333333492279053, 'Whistle': 0.8993902206420898}, 
AUC: {'Labels_Average': 0.9665114283561707, 'ECHO': 0.97560715675354, 'HFPC': 0.9787186980247498, 'BBPC': 0.9431157112121582, 'Whistle': 0.9686040878295898}, 
exact_match: {'Labels_Average': 0.7990867495536804},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  \
0

,Filename,Site,ECHO_true,ECHO_pred,ECHO_probs,HFPC_true,HFPC_pred,HFPC_probs,BBPC_true,BBPC_pred,BBPC_probs,Whistle_true,Whistle_pred,Whistle_probs
0,CAC_20210714_08001900.pt,CAC,1.0,1.0,9.988313e-01,0.0,0.0,4.621405e-06,0.0,0.0,1.473835e-05,1.0,1.0,9.999706e-01
1,CAC_20210714_08002200.pt,CAC,1.0,1.0,9.999840e-01,0.0,0.0,5.877128e-08,0.0,0.0,1.216090e-06,1.0,1.0,9.999986e-01
2,CAC_20210714_08002900.pt,CAC,1.0,1.0,9.999999e-01,0.0,0.0,2.318224e-06,0.0,0.0,9.530001e-06,0.0,1.0,9.991649e-01
3,CAC_20210714_08004600.pt,CAC,1.0,1.0,9.999970e-01,0.0,0.0,2.255585e-05,1.0,1.0,9.988870e-01,0.0,1.0,9.998580e-01
4,CAC_20210714_08010100.pt,CAC,1.0,1.0,9.999998e-01,0.0,0.0,5.821046e-08,0.0,0.0,2.976617e-06,1.0,1.0,9.999988e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3282,CAC_20210804_13305400.pt,CAC,0.0,0.0,1.590195e-09,0.0,0.0,7.773283e-09,0.0,0.0,2.840779e-09,0.0,0.0,1.676628e-08
3283,CAC_20210804_13310000.pt,CAC,0.0,0.0,2.979843e-09,0.0,0.0,1.562524e-07,0.0,0.0,7.172871e-08,0.0,0.0,5.361160e-06
3284,CAC_20210804_13310100.pt,CAC,0.0,0.0,3.029321e-09,0.0,0.0,1.703050e-08,0.0,0.0,2.867706e-08,0.0,0.0,8.416702e-07
3285,CAC_20210804_13310200.pt,CAC,0.0,0.0,3.777263e-09,0.0,0.0,1.713626e-07,0.0,0.0,2.039830e-06,0.0,0.0,2.504894e-06


#### Running all MobileNet variants: layer depth & quantization 

In [6]:

n_layers_to_test = [8, 6,  10, 12,]  
# n_layers_to_test = [2, 4, 6, 8, 10, 12,]  
quantization_options = [False,True]

for n_layers in n_layers_to_test:
    for use_quantization in quantization_options:
        # Create run name based on parameters
        quant_suffix = "_qat" if use_quantization else ""
        run_name = f"mobile_net{quant_suffix}_{n_layers}_layers"
        
        print(f"\n{'='*80}")
        print(f"Running experiment: {run_name}")
        print(f"n_layers: {n_layers}, quantization: {use_quantization}")
        print(f"{'='*80}")

        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model_kwargs = {
            "pretrained": True,
            "n_layers": n_layers
        }

        if use_quantization:
            model_kwargs["qat"] = True
        
        try:
            run_cross_val(
                labels_df, 
                label_columns, 
                model_class,  
                processed_spects_dir,
                run_name=run_name,
                model_kwargs=model_kwargs, 
                n_splits=5,
                training_config=training_config_default,
                save_models=True,
                use_quantization=use_quantization,
            )
            print(f"✅ Successfully completed: {run_name}")
            
        except Exception as e:
            print(f"❌ Error in experiment {run_name}: {str(e)}")
            print(f"Continuing with next experiment...")
            continue

print(f"\n{'='*80}")
print("All experiments completed!")
print(f"{'='*80}")


Val Epoch: 25


100%|██████████| 78/78 [00:09<00:00,  7.86it/s]


Val Epoch: 25 Results - 
loss: 18.293, 
accuracy: {'Labels_Average': 0.9726137518882751, 'ECHO': 0.9629480242729187, 'HFPC': 0.9855014085769653, 'BBPC': 0.9810712933540344, 'Whistle': 0.9609343409538269}, 
f1: {'Labels_Average': 0.8788601756095886, 'ECHO': 0.9471871256828308, 'HFPC': 0.843478262424469, 'BBPC': 0.8303248882293701, 'Whistle': 0.8944504857063293}, 
precision: {'Labels_Average': 0.8883719444274902, 'ECHO': 0.9482758641242981, 'HFPC': 0.843478262424469, 'BBPC': 0.8778625726699829, 'Whistle': 0.8838709592819214}, 
recall: {'Labels_Average': 0.8706341981887817, 'ECHO': 0.9461008906364441, 'HFPC': 0.843478262424469, 'BBPC': 0.7876712083816528, 'Whistle': 0.9052863717079163}, 
AUC: {'Labels_Average': 0.9874508380889893, 'ECHO': 0.9902142286300659, 'HFPC': 0.9879093766212463, 'BBPC': 0.9862003326416016, 'Whistle': 0.9854792356491089}, 
exact_match: {'Labels_Average': 0.9065646529197693},

Training Epoch: 26


100%|██████████| 311/311 [00:46<00:00,  6.72it/s]


Train Epoch: 26 Results - 
loss: 0.456, 
accuracy: {'Labels_Average': 0.9988672733306885, 'ECHO': 0.9984897375106812, 'HFPC': 0.9989931583404541, 'BBPC': 0.9997986555099487, 'Whistle': 0.9981876611709595}, 
f1: {'Labels_Average': 0.9953732490539551, 'ECHO': 0.997732400894165, 'HFPC': 0.990138053894043, 'BBPC': 0.9984301328659058, 'Whistle': 0.995192289352417}, 
precision: {'Labels_Average': 0.9951527118682861, 'ECHO': 0.9978832602500916, 'HFPC': 0.990138053894043, 'BBPC': 0.9968652129173279, 'Whistle': 0.9957242012023926}, 
recall: {'Labels_Average': 0.9955951571464539, 'ECHO': 0.9975816011428833, 'HFPC': 0.990138053894043, 'BBPC': 1.0, 'Whistle': 0.9946609735488892}, 
AUC: {'Labels_Average': 0.9999841451644897, 'ECHO': 0.9999843835830688, 'HFPC': 0.9999895691871643, 'BBPC': 0.9999959468841553, 'Whistle': 0.9999667406082153}, 
exact_match: {'Labels_Average': 0.9954692125320435},

Val Epoch: 26


...

(quantized) test on site rdl Epoch: 0 Results - 
loss: 51.184, 
accuracy: {'Labels_Average': 0.9166666865348816, 'ECHO': 0.9069767594337463, 'HFPC': 0.930232584476471, 'BBPC': 0.8914728760719299, 'Whistle': 0.9379844665527344}, 
f1: {'Labels_Average': 0.862175703048706, 'ECHO': 0.8666666746139526, 'HFPC': 0.8831169009208679, 'BBPC': 0.7586206793785095, 'Whistle': 0.9402984976768494}, 
precision: {'Labels_Average': 0.8604111671447754, 'ECHO': 0.8863636255264282, 'HFPC': 0.8292682766914368, 'BBPC': 0.7857142686843872, 'Whistle': 0.9402984976768494}, 
recall: {'Labels_Average': 0.8664755821228027, 'ECHO': 0.8478260636329651, 'HFPC': 0.9444444179534912, 'BBPC': 0.7333333492279053, 'Whistle': 0.9402984976768494}, 
AUC: {'Labels_Average': 0.9691227674484253, 'ECHO': 0.9566526412963867, 'HFPC': 0.9734169840812683, 'BBPC': 0.9661616086959839, 'Whistle': 0.9802598357200623}, 
exact_match: {'Labels_Average': 0.7441860437393188},
Created test dataloader for site: CAC with 657 samples

Test on sit

100%|██████████| 21/21 [00:02<00:00,  7.92it/s]


Test on site cac Epoch: 0 Results - 
loss: 36.316, 
accuracy: {'Labels_Average': 0.9406392574310303, 'ECHO': 0.9330289363861084, 'HFPC': 0.9726027250289917, 'BBPC': 0.95281583070755, 'Whistle': 0.9041095972061157}, 
f1: {'Labels_Average': 0.8473471403121948, 'ECHO': 0.9153845906257629, 'HFPC': 0.7804877758026123, 'BBPC': 0.7891156673431396, 'Whistle': 0.9044005870819092}, 
precision: {'Labels_Average': 0.8712427020072937, 'ECHO': 0.9370078444480896, 'HFPC': 0.8421052694320679, 'BBPC': 0.8055555820465088, 'Whistle': 0.9003021121025085}, 
recall: {'Labels_Average': 0.8259698748588562, 'ECHO': 0.8947368264198303, 'HFPC': 0.7272727489471436, 'BBPC': 0.7733333110809326, 'Whistle': 0.9085366129875183}, 
AUC: {'Labels_Average': 0.9623892307281494, 'ECHO': 0.971083402633667, 'HFPC': 0.9613674283027649, 'BBPC': 0.9525887370109558, 'Whistle': 0.96451735496521}, 
exact_match: {'Labels_Average': 0.8036529421806335},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  \
0

100%|██████████| 21/21 [00:02<00:00,  7.19it/s]


(quantized) test on site cac Epoch: 0 Results - 
loss: 40.183, 
accuracy: {'Labels_Average': 0.9379755854606628, 'ECHO': 0.9254185557365417, 'HFPC': 0.9726027250289917, 'BBPC': 0.9558599591255188, 'Whistle': 0.8980212807655334}, 
f1: {'Labels_Average': 0.8471972942352295, 'ECHO': 0.9052224159240723, 'HFPC': 0.7804877758026123, 'BBPC': 0.8053691387176514, 'Whistle': 0.8977099061012268}, 
precision: {'Labels_Average': 0.8710674047470093, 'ECHO': 0.9322709441184998, 'HFPC': 0.8421052694320679, 'BBPC': 0.8108108043670654, 'Whistle': 0.8990825414657593}, 
recall: {'Labels_Average': 0.8258283138275146, 'ECHO': 0.8796992301940918, 'HFPC': 0.7272727489471436, 'BBPC': 0.800000011920929, 'Whistle': 0.8963414430618286}, 
AUC: {'Labels_Average': 0.9596536159515381, 'ECHO': 0.9710882306098938, 'HFPC': 0.9504486918449402, 'BBPC': 0.9534821510314941, 'Whistle': 0.9635953903198242}, 
exact_match: {'Labels_Average': 0.7960426211357117},
✅ Successfully completed: mobile_net_qat_12_layers

All experiment

<h4>Function call template to run quick experiments<h4>


In [ ]:
run_cross_val(
    labels_df, 
    label_columns, 
    MobileNetMultilabel,  
    processed_spects_dir,
    run_name="mobile_net_hp_1024_8_layers_all_absences",
    model_kwargs={
        "pretrained":True,
        "n_layers": 8
    }, 
    n_splits=5,
    training_config=training_config_default,
    save_models=True,
    use_quantization=False,
)

## Site Generalization Experiments


1. **Site-specific models** — train a separate model per site.
2. **Leave-One-Site-Out** — train on all but one site and test on the held-out site to assess generalizability.

All runs follow the protocol described in the paper.


In [11]:
from training.cross_validation import create_test_fold_indices
from sklearn.model_selection import KFold, train_test_split
from models.utils import aggregate_folds_testing_metrics



labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")
labels_df["ClipFilenamePt"] = labels_df["clip_filename"].str.replace(".wav", ".pt", regex=False)


label_columns = ["ECHO", "HFPC", "BBPC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Verified_Dataset/spectrograms/"

results_dir = "./results/new_dataset"

labels_df = create_test_fold_indices(labels_df, 5)

### Site-specific models

In [14]:
all_sites = ["RDL", "CAC", "BSM", "KAM" ]

use_quantization = True

for train_site in all_sites:

    for fold_idx in range(5):
        
        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model = model_class(
            pretrained=True,
            n_layers=8,
            num_classes=len(label_columns)
        )

        train_site_df = labels_df[labels_df["Site"]==train_site]
        train_data = train_site_df[train_site_df["test_fold_idx"] != fold_idx]
        train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['Site'])
        
        test_data = train_site_df[train_site_df["test_fold_idx"] == fold_idx]

        run_name = f"{train_site}_only"
        if use_quantization:
            run_name = run_name + "_qat"


        run_dir, _, _ = train_model(
            labels_df,
            label_columns,
            model,
            train_data,
            val_data,
            test_data,
            processed_spects_dir=processed_spects_dir,
            fold_idx=fold_idx,
            results_dir="./results/sites_generalization",
            run_name=run_name,
            training_config=training_config_default,
            use_quantization=use_quantization,
            compute_sites_metrics=True
        )

    aggregate_folds_testing_metrics(run_dir)


Val Epoch: 30


100%|██████████| 19/19 [00:02<00:00,  7.91it/s]


Val Epoch: 30 Results - 
loss: 16.770, 
accuracy: {'Labels_Average': 0.9637436866760254, 'ECHO': 0.9392917156219482, 'HFPC': 0.9814502596855164, 'BBPC': 0.9679595232009888, 'Whistle': 0.9662731885910034}, 
f1: {'Labels_Average': 0.8701298236846924, 'ECHO': 0.9586206674575806, 'HFPC': 0.8070175647735596, 'BBPC': 0.8190476298332214, 'Whistle': 0.8958333134651184}, 
precision: {'Labels_Average': 0.8686857223510742, 'ECHO': 0.9564220309257507, 'HFPC': 0.7666666507720947, 'BBPC': 0.8269230723381042, 'Whistle': 0.9247311949729919}, 
recall: {'Labels_Average': 0.8731722831726074, 'ECHO': 0.960829496383667, 'HFPC': 0.8518518805503845, 'BBPC': 0.8113207817077637, 'Whistle': 0.868686854839325}, 
AUC: {'Labels_Average': 0.9809310436248779, 'ECHO': 0.9807770252227783, 'HFPC': 0.987174391746521, 'BBPC': 0.9762404561042786, 'Whistle': 0.9795321226119995}, 
exact_match: {'Labels_Average': 0.8718380928039551},
No improvement over last 5 epochs in validation loss. Early stopping...
Training complete. L

100%|██████████| 24/24 [00:03<00:00,  7.14it/s]


Test Epoch: 0 Results - 
loss: 17.848, 
accuracy: {'Labels_Average': 0.9612010717391968, 'ECHO': 0.9406207799911499, 'HFPC': 0.9824561476707458, 'BBPC': 0.9689608812332153, 'Whistle': 0.9527665376663208}, 
f1: {'Labels_Average': 0.8562071919441223, 'ECHO': 0.9622641801834106, 'HFPC': 0.7936508059501648, 'BBPC': 0.800000011920929, 'Whistle': 0.8689138293266296}, 
precision: {'Labels_Average': 0.88475501537323, 'ECHO': 0.9589743614196777, 'HFPC': 0.8333333134651184, 'BBPC': 0.8679245114326477, 'Whistle': 0.8787878751754761}, 
recall: {'Labels_Average': 0.8310867547988892, 'ECHO': 0.9655765891075134, 'HFPC': 0.7575757503509521, 'BBPC': 0.7419354915618896, 'Whistle': 0.8592592477798462}, 
AUC: {'Labels_Average': 0.9697461128234863, 'ECHO': 0.976220965385437, 'HFPC': 0.9725217819213867, 'BBPC': 0.9612095952033997, 'Whistle': 0.9690319299697876}, 
exact_match: {'Labels_Average': 0.8636977076530457},
Final test loss: 17.8484
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  

...

(quantized) test on site bsm Epoch: 0 Results - 
loss: 16.818, 
accuracy: {'Labels_Average': 0.9570114016532898, 'ECHO': 0.8965736031532288, 'HFPC': 0.9835025668144226, 'BBPC': 0.9809644818305969, 'Whistle': 0.9670050740242004}, 
f1: {'Labels_Average': 0.6743412017822266, 'ECHO': 0.7104795575141907, 'HFPC': 0.7868852615356445, 'BBPC': 0.5714285969734192, 'Whistle': 0.6285714507102966}, 
precision: {'Labels_Average': 0.6090647578239441, 'ECHO': 0.5763688683509827, 'HFPC': 0.7384615540504456, 'BBPC': 0.5714285969734192, 'Whistle': 0.550000011920929}, 
recall: {'Labels_Average': 0.7681982517242432, 'ECHO': 0.9259259104728699, 'HFPC': 0.8421052694320679, 'BBPC': 0.5714285969734192, 'Whistle': 0.7333333492279053}, 
AUC: {'Labels_Average': 0.9736206531524658, 'ECHO': 0.9708197116851807, 'HFPC': 0.9867121577262878, 'BBPC': 0.9610549211502075, 'Whistle': 0.9758959412574768}, 
exact_match: {'Labels_Average': 0.8578680157661438},
Created test dataloader for site: CAC with 657 samples

Test on si

100%|██████████| 21/21 [00:02<00:00,  7.10it/s]


Test on site cac Epoch: 0 Results - 
loss: 58.971, 
accuracy: {'Labels_Average': 0.8732876777648926, 'ECHO': 0.8599695563316345, 'HFPC': 0.9726027250289917, 'BBPC': 0.9238964915275574, 'Whistle': 0.7366818785667419}, 
f1: {'Labels_Average': 0.7287435531616211, 'ECHO': 0.8380281925201416, 'HFPC': 0.7804877758026123, 'BBPC': 0.6376811861991882, 'Whistle': 0.658777117729187}, 
precision: {'Labels_Average': 0.8153895735740662, 'ECHO': 0.7880794405937195, 'HFPC': 0.8421052694320679, 'BBPC': 0.6984127163887024, 'Whistle': 0.9329608678817749}, 
recall: {'Labels_Average': 0.6794556379318237, 'ECHO': 0.8947368264198303, 'HFPC': 0.7272727489471436, 'BBPC': 0.5866666436195374, 'Whistle': 0.5091463327407837}, 
AUC: {'Labels_Average': 0.9124950766563416, 'ECHO': 0.9418110251426697, 'HFPC': 0.9786074161529541, 'BBPC': 0.8796563744544983, 'Whistle': 0.8499054908752441}, 
exact_match: {'Labels_Average': 0.5799086689949036},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true 

100%|██████████| 21/21 [00:03<00:00,  6.81it/s]

(quantized) test on site cac Epoch: 0 Results - 
loss: 61.008, 
accuracy: {'Labels_Average': 0.8694825172424316, 'ECHO': 0.8477929830551147, 'HFPC': 0.9756468534469604, 'BBPC': 0.9193302989006042, 'Whistle': 0.7351598143577576}, 
f1: {'Labels_Average': 0.7282401323318481, 'ECHO': 0.8245614171028137, 'HFPC': 0.8095238208770752, 'BBPC': 0.6241135001182556, 'Whistle': 0.6547619104385376}, 
precision: {'Labels_Average': 0.8067982792854309, 'ECHO': 0.7730262875556946, 'HFPC': 0.8500000238418579, 'BBPC': 0.6666666865348816, 'Whistle': 0.9375}, 
recall: {'Labels_Average': 0.6864752769470215, 'ECHO': 0.88345867395401, 'HFPC': 0.7727272510528564, 'BBPC': 0.5866666436195374, 'Whistle': 0.5030487775802612}, 
AUC: {'Labels_Average': 0.9056947827339172, 'ECHO': 0.9408735632896423, 'HFPC': 0.9786445498466492, 'BBPC': 0.8682474493980408, 'Whistle': 0.8350136876106262}, 
exact_match: {'Labels_Average': 0.5753424763679504},


### Leave One Site out

In [15]:


all_sites = ["BSM", "RDL", "CAC", "KAM" ]

use_quantization = True

for out_site in all_sites:
    for fold_idx in range(5):
        
        model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

        model = model_class(
            pretrained=True,
            n_layers=8,
            num_classes=len(label_columns)
        )

        train_sites = [site for site in all_sites if site != out_site]

        train_df = labels_df[labels_df["test_fold_idx"] != fold_idx]
        
        train_sites_df = train_df[train_df["Site"].isin(train_sites)]

        train_data, val_data = train_test_split(train_sites_df, test_size=0.2, random_state=42, stratify=train_sites_df['Site'])
        
        test_data = labels_df[labels_df["test_fold_idx"] == fold_idx]


        run_name = f"leave_{out_site}_out"
        if use_quantization:
            run_name = run_name + "_qat"

        print(run_name)
        print(f"Site out : {out_site}")
        print("Train df")
        print(train_data["Site"].value_counts())
        print("\nVal df")
        print(val_data["Site"].value_counts())

        # break
        run_dir, _, _ = train_model(
            labels_df,
            label_columns,
            model,
            train_data,
            val_data,
            test_data,
            processed_spects_dir=processed_spects_dir,
            fold_idx=fold_idx,
            run_name=run_name,
            results_dir="./results/sites_generalization",
            training_config=training_config_default,
            use_quantization=use_quantization,
            
        )
    
    aggregate_folds_testing_metrics(run_dir)

    # break


Val Epoch: 15


100%|██████████| 60/60 [00:06<00:00,  9.05it/s]


Val Epoch: 15 Results - 
loss: 8.944, 
accuracy: {'Labels_Average': 0.9742199778556824, 'ECHO': 0.9666842818260193, 'HFPC': 0.9878371357917786, 'BBPC': 0.9830777645111084, 'Whistle': 0.959280788898468}, 
f1: {'Labels_Average': 0.8768349885940552, 'ECHO': 0.9191271066665649, 'HFPC': 0.8670520186424255, 'BBPC': 0.8279569745063782, 'Whistle': 0.893203854560852}, 
precision: {'Labels_Average': 0.8903688192367554, 'ECHO': 0.934725821018219, 'HFPC': 0.8928571343421936, 'BBPC': 0.8369565010070801, 'Whistle': 0.8969359397888184}, 
recall: {'Labels_Average': 0.8638471961021423, 'ECHO': 0.9040403962135315, 'HFPC': 0.8426966071128845, 'BBPC': 0.8191489577293396, 'Whistle': 0.889502763748169}, 
AUC: {'Labels_Average': 0.9925904273986816, 'ECHO': 0.9910753965377808, 'HFPC': 0.9952892065048218, 'BBPC': 0.9904450178146362, 'Whistle': 0.9935519695281982}, 
exact_match: {'Labels_Average': 0.9143310189247131},

Training Epoch: 16


100%|██████████| 237/237 [00:31<00:00,  7.61it/s]


Train Epoch: 16 Results - 
loss: 3.651, 
accuracy: {'Labels_Average': 0.9895502328872681, 'ECHO': 0.983730137348175, 'HFPC': 0.9964285492897034, 'BBPC': 0.9948412775993347, 'Whistle': 0.9832010865211487}, 
f1: {'Labels_Average': 0.9586455225944519, 'ECHO': 0.9604119658470154, 'HFPC': 0.9652509689331055, 'BBPC': 0.9527272582054138, 'Whistle': 0.9561917781829834}, 
precision: {'Labels_Average': 0.9695985913276672, 'ECHO': 0.9726206064224243, 'HFPC': 0.9689922332763672, 'BBPC': 0.9776119589805603, 'Whistle': 0.9591695666313171}, 
recall: {'Labels_Average': 0.9480887651443481, 'ECHO': 0.9485060572624207, 'HFPC': 0.9615384340286255, 'BBPC': 0.9290780425071716, 'Whistle': 0.95323246717453}, 
AUC: {'Labels_Average': 0.998546838760376, 'ECHO': 0.9973838925361633, 'HFPC': 0.9994990229606628, 'BBPC': 0.9990732073783875, 'Whistle': 0.9982312917709351}, 
exact_match: {'Labels_Average': 0.9611111283302307},

Val Epoch: 16


...

(quantized) test on site bsm Epoch: 0 Results - 
loss: 5.787, 
accuracy: {'Labels_Average': 0.9857233166694641, 'ECHO': 0.9727157354354858, 'HFPC': 0.9892131686210632, 'BBPC': 0.9930202960968018, 'Whistle': 0.9879441857337952}, 
f1: {'Labels_Average': 0.8619682788848877, 'ECHO': 0.8992974162101746, 'HFPC': 0.8682170510292053, 'BBPC': 0.8253968358039856, 'Whistle': 0.8549618124961853}, 
precision: {'Labels_Average': 0.8512585163116455, 'ECHO': 0.9099525809288025, 'HFPC': 0.7777777910232544, 'BBPC': 0.9285714030265808, 'Whistle': 0.7887324094772339}, 
recall: {'Labels_Average': 0.8868839144706726, 'ECHO': 0.8888888955116272, 'HFPC': 0.9824561476707458, 'BBPC': 0.7428571581840515, 'Whistle': 0.9333333373069763}, 
AUC: {'Labels_Average': 0.995177149772644, 'ECHO': 0.992594301700592, 'HFPC': 0.9973263144493103, 'BBPC': 0.9954110383987427, 'Whistle': 0.9953770637512207}, 
exact_match: {'Labels_Average': 0.9517766237258911},
Created test dataloader for site: CAC with 657 samples

Test on site

100%|██████████| 21/21 [00:02<00:00,  8.31it/s]


Test on site cac Epoch: 0 Results - 
loss: 23.366, 
accuracy: {'Labels_Average': 0.9330288767814636, 'ECHO': 0.9254185557365417, 'HFPC': 0.9710806608200073, 'BBPC': 0.9406392574310303, 'Whistle': 0.8949771523475647}, 
f1: {'Labels_Average': 0.835595965385437, 'ECHO': 0.9041095972061157, 'HFPC': 0.791208803653717, 'BBPC': 0.7483870983123779, 'Whistle': 0.8986784219741821}, 
precision: {'Labels_Average': 0.8251675367355347, 'ECHO': 0.9428571462631226, 'HFPC': 0.7659574747085571, 'BBPC': 0.7250000238418579, 'Whistle': 0.8668555021286011}, 
recall: {'Labels_Average': 0.8482157588005066, 'ECHO': 0.8684210777282715, 'HFPC': 0.8181818127632141, 'BBPC': 0.7733333110809326, 'Whistle': 0.9329268336296082}, 
AUC: {'Labels_Average': 0.9681711196899414, 'ECHO': 0.9804723262786865, 'HFPC': 0.9622015357017517, 'BBPC': 0.9614204168319702, 'Whistle': 0.9685901403427124}, 
exact_match: {'Labels_Average': 0.7686453461647034},
                   Filename Site  ECHO_true  ECHO_pred  ECHO_probs  HFPC_true  

100%|██████████| 21/21 [00:02<00:00,  7.76it/s]

(quantized) test on site cac Epoch: 0 Results - 
loss: 25.230, 
accuracy: {'Labels_Average': 0.9318873286247253, 'ECHO': 0.9284626841545105, 'HFPC': 0.9726027250289917, 'BBPC': 0.9391171932220459, 'Whistle': 0.8873668313026428}, 
f1: {'Labels_Average': 0.8388818502426147, 'ECHO': 0.90873783826828, 'HFPC': 0.804347813129425, 'BBPC': 0.75, 'Whistle': 0.8924418687820435}, 
precision: {'Labels_Average': 0.8173130750656128, 'ECHO': 0.9397590160369873, 'HFPC': 0.7708333134651184, 'BBPC': 0.7058823704719543, 'Whistle': 0.8527777791023254}, 
recall: {'Labels_Average': 0.8641459345817566, 'ECHO': 0.8796992301940918, 'HFPC': 0.8409090638160706, 'BBPC': 0.800000011920929, 'Whistle': 0.9359756112098694}, 
AUC: {'Labels_Average': 0.9634600877761841, 'ECHO': 0.9790156483650208, 'HFPC': 0.9524692296981812, 'BBPC': 0.956735372543335, 'Whistle': 0.9656201601028442}, 
exact_match: {'Labels_Average': 0.7625570893287659},


<h2>Training the final model on all the data</h2>

In [ ]:
from training.cross_validation import create_test_fold_indices
from sklearn.model_selection import KFold, train_test_split
from models.utils import aggregate_folds_testing_metrics



labels_df = pd.read_csv("../data/labels/Overlaps_1s.csv")
labels_df["ClipFilenamePt"] = labels_df["ClipFilename"] + ".pt"


label_columns = ["ECHO", "HFPC", "CC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Full_Dataset/Overlaps_1s_hp_1024_resize/"

results_dir = "./final_results"

labels_df = create_test_fold_indices(labels_df, 5)

In [ ]:

use_quantization = True
        
model_class = load_mobilenet_v3_quant if use_quantization else MobileNetMultilabel

model = model_class(
    pretrained=True,
    n_layers=8,
    num_classes=len(label_columns)
)

train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42, stratify=train_data['Site'])
test_data = val_data #Doesn't matter here, won't be used anyway

run_name = f"Final_model"
if use_quantization:
    run_name = run_name + "_qat"

run_dir = train_model(
    labels_df,
    label_columns,
    model,
    train_data,
    val_data,
    test_data,
    fold_idx=0,
    processed_spects_dir=processed_spects_dir,
    run_name=run_name,
    results_dir="results/final_model",
    training_config=training_config_default,
    use_quantization=use_quantization,
    save_model=True
)